# Robust Mean Variance Choice

## A Declared Set of Possible Means

Let the uncertain mean vector have the form

$$
\mu(u)=\widehat\mu+Lu,
\qquad
\lVert u\rVert_2\leq\rho.
$$

Here, $\widehat\mu$ is the supplied center, $L$ is a supplied loading matrix, $u$ is an unknown disturbance, and $\rho$ is the declared radius. For fixed portfolio weights $w$, the least favorable mean is

$$
\min_{\lVert u\rVert_2\leq\rho}w^{\mathsf T}\mu(u)
=
\widehat\mu^{\mathsf T}w
-
\rho\lVert L^{\mathsf T}w\rVert_2.
$$

The identity follows from the Cauchy Schwarz inequality, with equality in the direction opposite to $L^{\mathsf T}w$. The radius and loading are sensitivity inputs. They are not treated as a calibrated confidence region.

In [1]:
from io import StringIO
from itertools import combinations
from pathlib import Path
from zipfile import ZipFile
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
import RiskLabAI
from RiskLabAI.optimization import robust_mean_variance, spo_plus_loss, spo_plus_torch
assert RiskLabAI.__version__ == '3.2.0'
DATA_PATH = ensure_dataset(BOOK_ROOT, 'french_49_industries')
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '27_decision_focused_learning'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '27_decision_focused_learning'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '27_decision_focused_learning'
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (bool, np.bool_)):
        return 'Pass' if bool(value) else 'Fail'
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return '0'
        if number.is_integer():
            return f'{int(number):,}'
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f'{number:.6g}'
        return f'{number:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    display(figure)

def load_public_returns():
    assert DATA_PATH.is_file()
    with ZipFile(DATA_PATH) as archive:
        member = archive.namelist()[0]
        lines = archive.read(member).decode('latin1').splitlines()
    title_index = next((index for index, line in enumerate(lines) if 'Average Value Weighted Returns -- Daily' in line))
    header_index = title_index + 1
    end_index = next((index for index in range(header_index + 1, len(lines)) if not lines[index].strip()))
    frame = pd.read_csv(StringIO('\n'.join(lines[header_index:end_index])))
    frame = frame.rename(columns={frame.columns[0]: 'Date'})
    frame['Date'] = pd.to_datetime(frame['Date'].astype(str), format='%Y%m%d')
    industries = list(frame.columns[1:17])
    frame[industries] = frame[industries].apply(pd.to_numeric, errors='coerce')
    frame[industries] = frame[industries].mask(frame[industries] <= -99)
    frame = frame.loc[(frame['Date'] >= '2019-01-01') & (frame['Date'] <= '2025-12-31'), ['Date', *industries]].dropna().reset_index(drop=True)
    assert len(industries) == 16
    assert len(frame) == 1760
    return (frame, industries)
center = np.array([0.08, 0.12])
loading_small = np.array([[0.035, 0.012], [0.0, 0.05]])
weights_small = np.array([0.4, 0.6])
radius_small = 1.5
loading_exposure = loading_small.T @ weights_small
analytic_direction = -loading_exposure / np.linalg.norm(loading_exposure)
analytic_worst = float(center @ weights_small - radius_small * np.linalg.norm(loading_exposure))
angles = np.linspace(0, 2 * np.pi, 20001)
directions = np.column_stack([np.cos(angles), np.sin(angles)])
possible_means = center + radius_small * directions @ loading_small.T
portfolio_means = possible_means @ weights_small
grid_index = int(np.argmin(portfolio_means))
grid_worst = float(portfolio_means[grid_index])
notation = pd.DataFrame([{'Symbol': 'mu hat', 'Description': 'Supplied center of the possible mean vectors'}, {'Symbol': 'L', 'Description': 'Supplied loading matrix for mean uncertainty'}, {'Symbol': 'u', 'Description': 'Disturbance with Euclidean norm at most rho'}, {'Symbol': 'rho', 'Description': 'Declared uncertainty radius'}, {'Symbol': 'gamma', 'Description': 'Risk aversion multiplying portfolio variance'}])
save_table(notation, 'robust_choice_notation', 'Notation for robust mean variance choice.', 'tab:chapter28-robust-notation', fit=True)
support_checks = pd.DataFrame([{'Quantity': 'Nominal portfolio mean', 'Value': center @ weights_small, 'Pass': True}, {'Quantity': 'Uncertainty exposure norm', 'Value': np.linalg.norm(loading_exposure), 'Pass': True}, {'Quantity': 'Analytic least favorable mean', 'Value': analytic_worst, 'Pass': True}, {'Quantity': 'Finite angle search minimum', 'Value': grid_worst, 'Pass': abs(grid_worst - analytic_worst) < 1e-08}, {'Quantity': 'Direction norm', 'Value': np.linalg.norm(analytic_direction), 'Pass': np.isclose(np.linalg.norm(analytic_direction), 1)}])
assert support_checks['Pass'].all()
save_table(support_checks, 'robust_support_function_checks', 'Analytic and finite-search checks for the least favorable mean.', 'tab:chapter28-support-checks')
geometry = pd.DataFrame({'Mean 1': possible_means[:, 0], 'Mean 2': possible_means[:, 1], 'Portfolio mean': portfolio_means})
geometry.to_csv(FIGURE_DATA / 'robust_uncertainty_geometry.csv', index=False)
figure, axis = plt.subplots(figsize=(7.3, 6.1), constrained_layout=True)
axis.plot(geometry['Mean 1'], geometry['Mean 2'], color=PALETTE['cyan'], linewidth=2.0, label='Boundary of possible means')
axis.scatter([center[0]], [center[1]], color=PALETTE['green'], s=70, label='Supplied center')
axis.scatter([possible_means[grid_index, 0]], [possible_means[grid_index, 1]], color=PALETTE['rose'], s=70, label='Least favorable for fixed weights')
axis.set_xlabel('Mean component 1')
axis.set_ylabel('Mean component 2')
axis.set_title('A Declared Elliptical Set of Possible Means')
axis.legend()
finish_axis(axis)
save_figure(figure, 'robust_uncertainty_geometry')


,Symbol,Description
0,mu hat,Supplied center of the possible mean vectors
1,L,Supplied loading matrix for mean uncertainty
2,u,Disturbance with Euclidean norm at most rho
3,rho,Declared uncertainty radius
4,gamma,Risk aversion multiplying portfolio variance


,Quantity,Value,Pass
0,Nominal portfolio mean,0.104,Pass
1,Uncertainty exposure norm,0.037511,Pass
2,Analytic least favorable mean,0.047734,Pass
3,Finite angle search minimum,0.047734,Pass
4,Direction norm,1,Pass


<Figure size 730x610 with 1 Axes>

## The Bounded Portfolio Problem

The robust portfolio solves

$$
\max_{w}
\left\{
\widehat\mu^{\mathsf T}w
-
\rho\lVert L^{\mathsf T}w\rVert_2
-
\frac{\gamma}{2}w^{\mathsf T}\Sigma w
\right\},
$$

subject to $\mathbf 1^{\mathsf T}w=1$, $0\leq w_i\leq0.15$, and no return floor in the primary study. The covariance matrix is $\Sigma$, and $\gamma=5$ is risk aversion. When $\rho=0$, the uncertainty penalty vanishes and the problem is the nominal mean variance problem.

### Measuring uncertainty by block resampling

For industry $i$, the loading scale is the block-resampling standard error of its annualized sample mean:

$$
L=\operatorname{diag}(s_{\mu,1},\ldots,s_{\mu,16}).
$$

Each $s_{\mu,i}$ is computed from three hundred fixed resamples of twenty-day return blocks using observations from 2019 through 2023. This preserves short stretches within sampled blocks. It supplies a transparent sensitivity scale, not a probability statement about the entire uncertainty set.

In [2]:
public, industries = load_public_returns()
returns = public.set_index('Date')[industries] / 100
design_returns = returns.loc[:'2023-12-31']
final_returns = returns.loc['2024-01-01':]
annual_mean = 252 * design_returns.mean().to_numpy()
annual_covariance = 252 * design_returns.cov().to_numpy()
block_length = 20
resamples = 300
rng = np.random.default_rng(2803)
array = design_returns.to_numpy()
resampled_means = []
blocks_needed = int(np.ceil(len(array) / block_length))
for _ in range(resamples):
    starts = rng.integers(0, len(array) - block_length + 1, size=blocks_needed)
    sample = np.concatenate([array[start:start + block_length] for start in starts], axis=0)[:len(array)]
    resampled_means.append(252 * sample.mean(axis=0))
mean_standard_error = np.std(resampled_means, axis=0, ddof=1)
loading = np.diag(mean_standard_error)
caps = np.full(len(industries), 0.15)
risk_aversion = 5.0
radii = [0.0, 0.5, 1.0, 2.0]
results = {radius: robust_mean_variance(annual_mean, annual_covariance, loading, radius=radius, risk_aversion=risk_aversion, caps=caps) for radius in radii}
provenance = pd.DataFrame([{'Item': 'Public source', 'Value': 'Kenneth R. French Data Library, 49 Industry Portfolios Daily'}, {'Item': 'Stored archive', 'Value': DATA_PATH.name}, {'Item': 'Design observations', 'Value': len(design_returns)}, {'Item': 'Final observations', 'Value': len(final_returns)}, {'Item': 'Download during execution', 'Value': 'None'}, {'Item': 'Simulated observations', 'Value': 'None'}])
save_table(provenance, 'robust_public_data_provenance', 'Public data provenance for the robust portfolio study.', 'tab:chapter28-robust-data-provenance', fit=True)
resampling_settings = pd.DataFrame([{'Setting': 'Design start', 'Value': design_returns.index.min().date().isoformat()}, {'Setting': 'Design end', 'Value': design_returns.index.max().date().isoformat()}, {'Setting': 'Resamples', 'Value': resamples}, {'Setting': 'Block length', 'Value': block_length}, {'Setting': 'Random seed', 'Value': 2803}, {'Setting': 'Primary risk aversion', 'Value': risk_aversion}, {'Setting': 'Weight cap', 'Value': 0.15}])
save_table(resampling_settings, 'robust_resampling_settings', 'Fixed estimation and resampling settings.', 'tab:chapter28-robust-resampling-settings')
uncertainty_table = pd.DataFrame({'Industry': industries, 'Annualized sample mean': annual_mean, 'Annualized volatility': np.sqrt(np.diag(annual_covariance)), 'Mean standard error': mean_standard_error})
save_table(uncertainty_table, 'robust_mean_uncertainty_scales', 'Design-period estimates and supplied mean uncertainty scales.', 'tab:chapter28-uncertainty-scales', fit=True)
weight_table = pd.DataFrame({'Industry': industries, **{f'Radius {radius:g}': results[radius]['weights'] for radius in radii}})
save_table(weight_table, 'robust_radius_weights', 'Portfolio weights across the declared uncertainty radii.', 'tab:chapter28-radius-weights', fit=True)
diagnostic_rows = []
for radius in radii:
    result = results[radius]
    weights = result['weights']
    diagnostic_rows.append({'Radius': radius, 'Nominal expected return': result['expected_return'], 'Worst permitted expected return': result['worst_case_expected_return'], 'Annualized volatility': np.sqrt(result['variance']), 'Squared concentration': np.sum(weights ** 2), 'Largest weight': weights.max(), 'Weight sum residual': abs(weights.sum() - 1), 'Cap residual': max(0.0, weights.max() - caps.max()), 'Solver status': result['solver_status']})
diagnostics = pd.DataFrame(diagnostic_rows)
assert np.allclose(diagnostics['Nominal expected return'] - diagnostics['Worst permitted expected return'], [radius * np.linalg.norm(loading.T @ results[radius]['weights']) for radius in radii])
assert diagnostics['Weight sum residual'].max() < 1e-07
assert diagnostics['Cap residual'].max() < 1e-07
save_table(diagnostics, 'robust_radius_diagnostics', 'Feasibility and objective components across uncertainty radii.', 'tab:chapter28-radius-diagnostics', fit=True)
heatmap_data = weight_table.set_index('Industry')
heatmap_data.to_csv(FIGURE_DATA / 'robust_radius_weights.csv')
figure, axis = plt.subplots(figsize=(8.3, 6.7), constrained_layout=True)
image = axis.imshow(heatmap_data.to_numpy(), cmap='magma', aspect='auto', vmin=0, vmax=0.15)
axis.set_xticks(np.arange(len(radii)), [f'Radius {radius:g}' for radius in radii])
axis.set_yticks(np.arange(len(industries)), industries)
axis.set_title('Portfolio Weights Change with the Declared Radius')
colorbar = figure.colorbar(image, ax=axis)
colorbar.set_label('Portfolio weight')
save_figure(figure, 'robust_radius_weight_heatmap')
penalty_data = diagnostics[['Radius', 'Nominal expected return', 'Worst permitted expected return']].copy()
penalty_data.to_csv(FIGURE_DATA / 'robust_radius_returns.csv', index=False)
figure, axis = plt.subplots(figsize=(9.2, 5.1), constrained_layout=True)
axis.plot(penalty_data['Radius'], penalty_data['Nominal expected return'], marker='o', color=PALETTE['green'], linewidth=2.0, label='Nominal mean')
axis.plot(penalty_data['Radius'], penalty_data['Worst permitted expected return'], marker='o', color=PALETTE['rose'], linewidth=2.0, label='Worst permitted mean')
axis.set_xlabel('Declared radius')
axis.set_ylabel('Annualized expected return')
axis.set_title('The Mean Uncertainty Penalty Grows with the Declared Radius')
axis.legend()
finish_axis(axis)
save_figure(figure, 'robust_radius_returns')
concentration_data = diagnostics[['Radius', 'Squared concentration', 'Largest weight']].copy()
concentration_data.to_csv(FIGURE_DATA / 'robust_radius_concentration.csv', index=False)
figure, axis = plt.subplots(figsize=(9.2, 5.1), constrained_layout=True)
axis.plot(concentration_data['Radius'], concentration_data['Squared concentration'], marker='o', color=PALETTE['cyan'], linewidth=2.0, label='Squared concentration')
axis.plot(concentration_data['Radius'], concentration_data['Largest weight'], marker='s', color=PALETTE['amber'], linewidth=2.0, label='Largest weight')
axis.set_xlabel('Declared radius')
axis.set_ylabel('Weight summary')
axis.set_title('Weight Concentration Is Reported, Not Presumed')
axis.legend()
finish_axis(axis)
save_figure(figure, 'robust_radius_concentration')


,Item,Value
0,Public source,"Kenneth R. French Data Library, 49 Industry Po..."
1,Stored archive,49_Industry_Portfolios_daily_CSV.zip
2,Design observations,"1,258"
3,Final observations,502
4,Download during execution,None
5,Simulated observations,None


,Setting,Value
0,Design start,2019-01-02
1,Design end,2023-12-29
2,Resamples,300
3,Block length,20
4,Random seed,"2,803"
5,Primary risk aversion,5
6,Weight cap,0.15


,Industry,Annualized sample mean,Annualized volatility,Mean standard error
0,Agric,0.183431,0.303278,0.105591
1,Food,0.092767,0.174609,0.064781
2,Soda,0.120832,0.207166,0.085726
3,Beer,0.122034,0.204604,0.075751
4,Smoke,0.116505,0.23306,0.095867
5,Toys,0.078705,0.380332,0.17363
6,Fun,0.129125,0.330232,0.144057
7,Books,0.142086,0.265756,0.120123
8,Hshld,0.112258,0.194684,0.077954
9,Clths,0.135275,0.306839,0.133385


,Industry,Radius 0,Radius 0.5,Radius 1,Radius 2
0,Agric,0.15,0.146231,0.120776,0.095299
1,Food,0.027201,0.099472,0.123077,0.140143
2,Soda,0.15,0.113562,0.103244,0.095841
3,Beer,0.15,0.120026,0.116118,0.114462
4,Smoke,0.065401,0.064944,0.068293,0.070168
5,Toys,2.56251e-12,1.8234e-12,1.87692e-12,4.71286e-12
6,Fun,1.91246e-10,0.007589,0.017624,0.024234
7,Books,0.087249,0.060539,0.055876,0.051189
8,Hshld,0.070149,0.087637,0.094991,0.100794
9,Clths,1.73685e-11,7.77188e-12,3.00997e-11,0.014525


,Radius,Nominal expected return,Worst permitted expected return,Annualized volatility,Squared concentration,Largest weight,Weight sum residual,Cap residual,Solver status
0,0,0.139115,0.139115,0.181546,0.13005,0.15,3.33067e-16,0,optimal
1,0.5,0.136162,0.120683,0.178742,0.119201,0.15,2.22045e-16,0,optimal
2,1,0.133808,0.103932,0.176947,0.115998,0.15,0,0,optimal
3,2,0.12906,0.074958,0.176196,0.104126,0.15,0,0,optimal


<Figure size 830x670 with 2 Axes>

<Figure size 920x510 with 1 Axes>

<Figure size 920x510 with 1 Axes>

## Checking the robust optimization contract

A nominal return floor, when supplied, constrains $\widehat\mu^{\mathsf T}w$. It does not constrain the worst permitted mean. The primary study uses no floor. Separate checks below show one feasible nominal floor and explicit rejection of an impossible floor.

In [3]:
zero_radius = results[0.0]
zero_loading = robust_mean_variance(annual_mean, annual_covariance, np.zeros_like(loading), radius=2.0, risk_aversion=risk_aversion, caps=caps)
identity_checks = pd.DataFrame([
    {"Check": "Radius zero equals zero-penalty weights", "Maximum residual": np.max(np.abs(zero_radius["weights"] - zero_loading["weights"])), "Pass": np.allclose(zero_radius["weights"], zero_loading["weights"], atol=1e-7)},
    {"Check": "Radius zero nominal mean equals worst mean", "Maximum residual": abs(zero_radius["expected_return"] - zero_radius["worst_case_expected_return"]), "Pass": np.isclose(zero_radius["expected_return"], zero_radius["worst_case_expected_return"])},
    {"Check": "All primary weights sum to one", "Maximum residual": max(abs(results[radius]["weights"].sum() - 1) for radius in radii), "Pass": all(abs(results[radius]["weights"].sum() - 1) < 1e-7 for radius in radii)},
    {"Check": "All primary weights obey the cap", "Maximum residual": max(max(0.0, results[radius]["weights"].max() - 0.15) for radius in radii), "Pass": all(results[radius]["weights"].max() <= 0.15 + 1e-7 for radius in radii)},
])
assert identity_checks["Pass"].all()
save_table(identity_checks, "robust_identity_and_feasibility_checks", "Radius-zero and primary feasibility checks.", "tab:chapter28-robust-identity-checks", fit=True)

equal_mean = float(annual_mean.mean())
feasible_floor = robust_mean_variance(annual_mean, annual_covariance, loading, radius=1.0, risk_aversion=risk_aversion, caps=caps, return_floor=equal_mean)
infeasible_rejected = False
try:
    robust_mean_variance(annual_mean, annual_covariance, loading, radius=1.0, risk_aversion=risk_aversion, caps=caps, return_floor=float(annual_mean.max() + 0.01))
except ValueError:
    infeasible_rejected = True
floor_checks = pd.DataFrame([
    {"Case": "Equal-weight nominal mean floor", "Floor": equal_mean, "Returned nominal mean": feasible_floor["expected_return"], "Pass": feasible_floor["expected_return"] >= equal_mean - 1e-7},
    {"Case": "Floor above every individual mean", "Floor": float(annual_mean.max() + 0.01), "Returned nominal mean": np.nan, "Pass": infeasible_rejected},
])
assert floor_checks["Pass"].all()
save_table(floor_checks, "robust_return_floor_checks", "Feasible and infeasible nominal return floor checks.", "tab:chapter28-return-floor-checks", fit=True)

sensitivity_rows = []
for gamma in [2.5, 5.0, 10.0]:
    for radius in radii:
        result = robust_mean_variance(annual_mean, annual_covariance, loading, radius=radius, risk_aversion=gamma, caps=caps)
        sensitivity_rows.append({"Risk aversion": gamma, "Radius": radius, "Nominal expected return": result["expected_return"], "Worst permitted expected return": result["worst_case_expected_return"], "Annualized volatility": np.sqrt(result["variance"]), "Squared concentration": np.sum(result["weights"] ** 2)})
sensitivity = pd.DataFrame(sensitivity_rows)
save_table(sensitivity, "robust_risk_aversion_sensitivity", "Sensitivity to risk aversion and uncertainty radius.", "tab:chapter28-risk-aversion-sensitivity", fit=True)
sensitivity.to_csv(FIGURE_DATA / "robust_risk_aversion_sensitivity.csv", index=False)
figure, axis = plt.subplots(figsize=(9.2, 5.1), constrained_layout=True)
for gamma, color in zip([2.5, 5.0, 10.0], [PALETTE["cyan"], PALETTE["amber"], PALETTE["green"]]):
    part = sensitivity.loc[sensitivity["Risk aversion"] == gamma]
    axis.plot(part["Radius"], part["Annualized volatility"], marker="o", linewidth=2.0, color=color, label=f"Risk aversion {gamma:g}")
axis.set_xlabel("Declared radius")
axis.set_ylabel("Annualized volatility from design covariance")
axis.set_title("Risk Aversion and Radius Are Separate Inputs")
axis.legend()
finish_axis(axis)
save_figure(figure, "robust_risk_aversion_sensitivity")

,Check,Maximum residual,Pass
0,Radius zero equals zero-penalty weights,3.38008e-09,Pass
1,Radius zero nominal mean equals worst mean,0,Pass
2,All primary weights sum to one,3.33067e-16,Pass
3,All primary weights obey the cap,0,Pass


,Case,Floor,Returned nominal mean,Pass
0,Equal-weight nominal mean floor,0.121378,0.133808,Pass
1,Floor above every individual mean,0.193431,,Pass


,Risk aversion,Radius,Nominal expected return,Worst permitted expected return,Annualized volatility,Squared concentration
0,2.5,0,0.141552,0.141552,0.184768,0.141123
1,2.5,0.5,0.14013,0.1237,0.183302,0.126762
2,2.5,1,0.138173,0.107224,0.182019,0.113545
3,2.5,2,0.131456,0.077397,0.179964,0.095703
4,5,0,0.139115,0.139115,0.181546,0.13005
5,5,0.5,0.136162,0.120683,0.178742,0.119201
6,5,1,0.133808,0.103932,0.176947,0.115998
7,5,2,0.12906,0.074958,0.176196,0.104126
8,10,0,0.130898,0.130898,0.174565,0.126471
9,10,0.5,0.130549,0.115775,0.174463,0.120883


<Figure size 920x510 with 1 Axes>

## Final Period Description

For fixed weights $w$, the final-period annualized mean and volatility are

$$
\overline r_{\mathrm{ann}}(w)=252\,\overline r(w),
\qquad
\sigma_{\mathrm{ann}}(w)=\sqrt{252}\,s(w).
$$

The quantity $\overline r(w)$ is the daily sample mean and $s(w)$ is the daily sample standard deviation during 2024 and 2025. These calculations describe one later period. They do not select a radius, estimate trading costs, or establish that one rule is generally better.

In [4]:
portfolio_weights = {"Equal weight": np.full(len(industries), 1 / len(industries)), **{f"Radius {radius:g}": results[radius]["weights"] for radius in radii}}
final_rows = []
for name, weights in portfolio_weights.items():
    series = final_returns.to_numpy() @ weights
    wealth = np.cumprod(1 + series)
    peak = np.maximum.accumulate(wealth)
    drawdown = wealth / peak - 1
    annual_return = 252 * series.mean()
    annual_volatility = np.sqrt(252) * series.std(ddof=1)
    final_rows.append({
        "Portfolio": name,
        "Annualized mean return": annual_return,
        "Annualized volatility": annual_volatility,
        "Mean to volatility ratio": annual_return / annual_volatility,
        "Maximum drawdown": drawdown.min(),
        "Turnover from equal weight": 0.5 * np.abs(weights - 1 / len(industries)).sum(),
        "Squared concentration": np.sum(weights ** 2),
    })
final_metrics = pd.DataFrame(final_rows)
save_table(final_metrics, "robust_final_period_metrics", "Descriptive results for fixed portfolios in 2024 and 2025.", "tab:chapter28-robust-final-metrics", fit=True)
final_metrics.to_csv(FIGURE_DATA / "robust_final_period_metrics.csv", index=False)
figure, axes = plt.subplots(1, 3, figsize=(12.0, 4.7), constrained_layout=True)
colors = [PALETTE["violet"], PALETTE["cyan"], PALETTE["amber"], PALETTE["green"], PALETTE["rose"]]
for axis, metric, title in zip(axes, ["Annualized mean return", "Annualized volatility", "Maximum drawdown"], ["Mean Return", "Volatility", "Maximum Drawdown"]):
    axis.bar(final_metrics["Portfolio"], final_metrics[metric], color=colors)
    axis.set_title(title)
    axis.tick_params(axis="x", rotation=55)
    finish_axis(axis)
save_figure(figure, "robust_final_period_metrics")

boundaries = pd.DataFrame([
    {"Question": "Does radius zero recover the nominal problem?", "Answered": True, "Boundary": "Checked numerically"},
    {"Question": "Do returned weights obey the cap and total?", "Answered": True, "Boundary": "Residuals below declared tolerance"},
    {"Question": "Is the loading a calibrated confidence region?", "Answered": False, "Boundary": "Sensitivity scale only"},
    {"Question": "Is a best radius selected from the final period?", "Answered": False, "Boundary": "No tuning from final outcomes"},
    {"Question": "Does one later period prove superiority?", "Answered": False, "Boundary": "Descriptive comparison only"},
])
save_table(boundaries, "robust_public_study_boundaries", "Questions answered and not answered by the robust portfolio study.", "tab:chapter28-robust-boundaries", fit=True)

,Portfolio,Annualized mean return,Annualized volatility,Mean to volatility ratio,Maximum drawdown,Turnover from equal weight,Squared concentration
0,Equal weight,0.082283,0.134535,0.611606,-0.149905,0,0.0625
1,Radius 0,0.095212,0.124271,0.766164,-0.116894,0.472799,0.13005
2,Radius 0.5,0.087063,0.123745,0.703569,-0.117678,0.431872,0.119201
3,Radius 1,0.082559,0.122077,0.67629,-0.118042,0.4265,0.115998
4,Radius 2,0.076337,0.119483,0.638894,-0.115085,0.372809,0.104126


<Figure size 1200x470 with 3 Axes>

,Question,Answered,Boundary
0,Does radius zero recover the nominal problem?,Pass,Checked numerically
1,Do returned weights obey the cap and total?,Pass,Residuals below declared tolerance
2,Is the loading a calibrated confidence region?,Fail,Sensitivity scale only
3,Is a best radius selected from the final period?,Fail,No tuning from final outcomes
4,Does one later period prove superiority?,Fail,Descriptive comparison only


,Question,Answered,Boundary
0,Does radius zero recover the nominal problem?,Pass,Checked numerically
1,Do returned weights obey the cap and total?,Pass,Residuals below declared tolerance
2,Is the loading a calibrated confidence region?,Fail,Sensitivity scale only
3,Is a best radius selected from the final period?,Fail,No tuning from final outcomes
4,Does one later period prove superiority?,Fail,Descriptive comparison only
